In [1]:
import pandas as pd
from pymatgen.core.periodic_table import Element
import math
import numpy as np

def get_data():
    """get data from file.

    Returns:
        pd.DataFrame: data.
    """
    columns_use = ['liquid_range',
                   'mineral_hardness', 'molar_volume', 'poissons_ratio',
                   'density_of_solid',  'critical_temperature', 'rigidity_modulus',
                   'brinell_hardness',
                   'boiling_point', 'bulk_modulus',
                   'X',
                   'superconduction_temperature', 'melting_point',
                   'velocity_of_sound', 'reflectivity',
                   'youngs_modulus']
    log_columns_use = [
                   'electrical_resistivity', 'thermal_conductivity',
                   'vickers_hardness',
                   'coefficient_of_linear_thermal_expansion']
    element_properteis = {}
    for Z in range(1,103):
        elm = Element("H").from_Z(Z)    
        properties = {}
        for prop in columns_use:
            try:
                value_unit = elm.__getattr__(prop)
                if value_unit is not None:
                    if isinstance(value_unit,str):
                        s = value_unit.split()
                        try:
                            value = float(s[0])
                        except ValueError:
                            value = None
                    else:
                        value = float(value_unit)
                    if value is not None:
                        properties[prop] = value
            except AttributeError:
                #print("no", prop)
                pass 
        for prop in log_columns_use:
            try:
                value_unit = elm.__getattr__(prop)
                if value_unit is not None:
                    value = float(value_unit)
                    value = math.log10(value)
                    properties["log10_"+prop] = value
            except AttributeError:
                # print("no", prop)]
                pass
        value = elm.X
        if value is not None and not np.isnan(value):
            properties["X"] = value
        element_properteis[str(elm)] = properties
    return element_properteis

g_properties =get_data()

/tmp/mi-in-place.WyFyuX/ipykernel_11751/892915910.py:31: UserWarning: No data available for mineral_hardness for H
  value_unit = elm.__getattr__(prop)
/tmp/mi-in-place.WyFyuX/ipykernel_11751/892915910.py:31: UserWarning: No data available for poissons_ratio for H
  value_unit = elm.__getattr__(prop)
/tmp/mi-in-place.WyFyuX/ipykernel_11751/892915910.py:31: UserWarning: No data available for density_of_solid for H
  value_unit = elm.__getattr__(prop)
/tmp/mi-in-place.WyFyuX/ipykernel_11751/892915910.py:31: UserWarning: No data available for rigidity_modulus for H
  value_unit = elm.__getattr__(prop)
/tmp/mi-in-place.WyFyuX/ipykernel_11751/892915910.py:31: UserWarning: No data available for brinell_hardness for H
  value_unit = elm.__getattr__(prop)
/tmp/mi-in-place.WyFyuX/ipykernel_11751/892915910.py:31: UserWarning: No data available for bulk_modulus for H
  value_unit = elm.__getattr__(prop)
/tmp/mi-in-place.WyFyuX/ipykernel_11751/892915910.py:31: UserWarning: No data available for su

In [2]:
import os
os.makedirs("data", exist_ok=True)
pd.DataFrame(g_properties).T.to_csv("data/atomicprop.csv")

In [3]:

def make_items(v, ndiv):
    """make a list of item names by digitize their values.

    Args:
        v (np.ndarray): data.
        ndiv (int): the number of divisions.

    Returns:
        [str]: item list.
    """
    vlist = []

    v = np.sort(v)
    len_v = v.shape[0]
    for d in range(1, ndiv):
        vlist.append(v[len_v*d//ndiv])
    
    itemlist = []
    # for value in vlist:
    #    for ineq in [">","<"]:
    #        itemlist.append("{}{:.2f}".format(ineq,value))
    itemlist.append("{}{:.2f}".format("<", vlist[0]))
    itemlist.append("{}{:.2f}".format(">", vlist[-1]))
    return itemlist

g_df = pd.DataFrame(g_properties).T

def make_min_max(df, ndiv=3):
    """make min and max. values.
    
    Args:
        properties (dict): element properties.
        
    Returns:
        dict: {element: (min, max)}
    
    """
    prop_minmax = {}
    for col in df.columns:
        values = df[col].values
        values = values[~np.isnan(values)]
        prop_minmax[col] = make_items(values, ndiv)
    return prop_minmax
                
g_prop_minmax = make_min_max(g_df)

def make_transaction(df2, itemlists):
    """make transaction, or query sentenses, and their results

    Args: 
        df2 (pd.DataFrame): data
        itemlists (list): a list of items

    Returns:
        list: transaction
    """
    labellist = []
    df_list = []
    lenindex = df2.shape[0]
    for prop in itemlists.keys():
        for item in itemlists[prop]:
            # print(prop,item)


            qstr = "{}{}".format(prop, item)
            
            v = np.array([False for i in range(lenindex)])
            df = pd.DataFrame(v[np.newaxis,:], 
                                columns=df2.index.tolist(), index=[qstr]).T
            
            idx = df2.query(qstr).index.tolist()
            df.loc[idx, qstr] = True

            df_list.append(df)
    df3 = pd.concat(df_list, axis=1)
    display(df3)
    
    transaction = {}
    for id_ in df3.index:
        transaction1 = []
        for col in df3.columns:
            flag = df3.loc[id_, col]
            if flag:
                transaction1.append(str(col))
        transaction[id_]=transaction1

    return transaction


g_transaction = make_transaction(g_df, g_prop_minmax)
g_transaction


,liquid_range<708.00,liquid_range>1816.00,molar_volume<12.29,molar_volume>19.10,critical_temperature<209.41,critical_temperature>1673.00,boiling_point<1757.00,boiling_point>3373.00,melting_point<903.78,melting_point>1629.00,...,poissons_ratio<0.26,poissons_ratio>0.31,brinell_hardness<294.00,brinell_hardness>736.00,superconduction_temperature<0.60,superconduction_temperature>1.70,log10_vickers_hardness<2.69,log10_vickers_hardness>3.02,reflectivity<65.00,reflectivity>74.00
H,True,False,True,False,True,False,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
He,True,False,False,True,True,False,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
Li,False,False,False,False,False,True,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
Be,False,False,True,False,False,False,False,False,False,False,...,True,False,False,False,True,False,False,True,False,False
B,False,True,True,False,False,False,False,True,False,True,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cf,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
Es,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
Fm,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
Md,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


{'H': ['liquid_range<708.00',
  'molar_volume<12.29',
  'critical_temperature<209.41',
  'boiling_point<1757.00',
  'melting_point<903.78',
  'velocity_of_sound<2310.00',
  'log10_thermal_conductivity<1.11',
  'X>2.00'],
 'He': ['liquid_range<708.00',
  'molar_volume>19.10',
  'critical_temperature<209.41',
  'boiling_point<1757.00',
  'melting_point<903.78',
  'velocity_of_sound<2310.00',
  'log10_thermal_conductivity<1.11'],
 'Li': ['critical_temperature>1673.00',
  'boiling_point<1757.00',
  'melting_point<903.78',
  'velocity_of_sound>3700.00',
  'log10_thermal_conductivity>1.73',
  'X<1.30',
  'mineral_hardness<2.25',
  'density_of_solid<6146.00',
  'rigidity_modulus<19.00',
  'bulk_modulus<33.00',
  'youngs_modulus<47.00',
  'log10_electrical_resistivity<-6.89',
  'log10_coefficient_of_linear_thermal_expansion>-4.87'],
 'Be': ['molar_volume<12.29',
  'velocity_of_sound>3700.00',
  'log10_thermal_conductivity>1.73',
  'density_of_solid<6146.00',
  'rigidity_modulus>43.00',
  'bulk

In [4]:
import json
import os
with open("../data_calculated/atom_transaction.json", "w") as f:
    json.dump(g_transaction,f)
